[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shuishen112/neural-networks/blob/main/lectures/07_rnns.ipynb)

# Lecture 7 — Recurrent Neural Networks

**Neural Networks · Master's course**

So far every input has been a fixed-size vector or image. Much of the world,
however, arrives as a **sequence**: text, speech, sensor readings, stock
prices, DNA. Sequences have variable length and their elements depend on one
another in order. In this lecture we will

1. describe the common sequence-modelling setups (many-to-one, many-to-many, seq2seq),
2. write a vanilla **recurrent neural network** (RNN) by hand and check it against `nn.RNN`,
3. derive **backpropagation through time** and see — on paper and numerically —
   why gradients vanish or explode,
4. fix exploding gradients with **gradient clipping**,
5. fix vanishing gradients with gated cells: the **LSTM** and the **GRU**,
6. compare RNN and LSTM on a long-range **memory task**,
7. train a **character-level language model** and sample from it with a temperature, and
8. cover the practicalities: padding, packing and bidirectional RNNs.

Recurrent networks have largely been replaced by Transformers (Lecture 8) for
large-scale language modelling, but the ideas introduced here — hidden state,
gating, additive paths for gradients — are everywhere in modern deep learning
(residual networks, state-space models, and even the analysis of Transformers).

In [ ]:
import math
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
rng = np.random.default_rng(0)
torch.set_num_threads(2)
plt.rcParams["figure.figsize"] = (5, 4)
plt.rcParams["axes.grid"] = True
print("torch", torch.__version__)

## 1. Sequence modelling setups

Let $\mathbf{x}_{1:T} = (\mathbf{x}_1, \dots, \mathbf{x}_T)$ with $\mathbf{x}_t \in \mathbb{R}^d$
be an input sequence. Depending on what we want to predict we distinguish

| Setup | Input → output | Examples |
|---|---|---|
| **many-to-one** | $\mathbf{x}_{1:T} \mapsto y$ | sentiment of a review, activity recognition from a sensor trace |
| **many-to-many (aligned)** | $\mathbf{x}_{1:T} \mapsto y_{1:T}$ | part-of-speech tagging, next-character prediction (language modelling) |
| **sequence-to-sequence** | $\mathbf{x}_{1:T} \mapsto y_{1:T'}$, $T \ne T'$ in general | translation, summarisation, speech recognition |
| **one-to-many** | $\mathbf{x} \mapsto y_{1:T}$ | image captioning, sampling text from a prompt |

A feed-forward network could handle a fixed-length window, but it would need
separate weights for every position and could not share what it learns about
"a pattern at position 3" with "the same pattern at position 17". The key idea of a
recurrent network is **weight sharing across time**: the same function is applied at
every step, carrying a summary of the past in a **hidden state** $\mathbf{h}_t$.
This is to sequences what convolution (Lecture 6) is to images.

## 2. The vanilla (Elman) RNN

The Elman RNN updates its hidden state $\mathbf{h}_t \in \mathbb{R}^H$ by

$$\mathbf{h}_t = \tanh\big(W_{ih}\,\mathbf{x}_t + \mathbf{b}_{ih} + W_{hh}\,\mathbf{h}_{t-1} + \mathbf{b}_{hh}\big),
\qquad \mathbf{h}_0 = \mathbf{0},$$

with $W_{ih} \in \mathbb{R}^{H\times d}$ and $W_{hh}\in\mathbb{R}^{H\times H}$. (PyTorch keeps two
bias vectors for compatibility with cuDNN; mathematically only their sum matters.)
Outputs are read from the hidden states with an ordinary layer,
e.g. $\hat{\mathbf{y}}_t = \operatorname{softmax}(W_{hy}\mathbf{h}_t + \mathbf{b}_y)$.

"Unrolling" the recurrence over time turns the RNN into a very deep feed-forward
network with $T$ layers that all **share the same weights**:

```
   x_1        x_2        x_3              x_T
    |          |          |                |
 h_0 -> [cell] -> h_1 -> [cell] -> h_2 -> ... -> [cell] -> h_T
                  |                 |                      |
                 y_1               y_2                    y_T
```

Let us implement the forward pass by hand and check it against `nn.RNN`. With
`batch_first=True`, PyTorch expects inputs of shape `(B, T, d)` and returns

* `output` of shape `(B, T, H)` — the hidden state at every step (top layer), and
* `h_n` of shape `(num_layers * num_directions, B, H)` — the final hidden state of every layer.

In [ ]:
def rnn_forward(x, W_ih, W_hh, b_ih, b_hh, h0=None):
    """Vanilla tanh RNN. x: (B, T, d). Returns all states (B, T, H) and the last state (B, H)."""
    B, T, _ = x.shape
    H = W_hh.shape[0]
    h = x.new_zeros(B, H) if h0 is None else h0
    states = []
    for t in range(T):
        h = torch.tanh(x[:, t] @ W_ih.T + b_ih + h @ W_hh.T + b_hh)
        states.append(h)
    return torch.stack(states, dim=1), h

d_in, H, B, T = 3, 5, 2, 7
rnn = nn.RNN(input_size=d_in, hidden_size=H, batch_first=True)
x = torch.randn(B, T, d_in)

with torch.no_grad():
    out_ref, h_ref = rnn(x)
    out_me, h_me = rnn_forward(x, rnn.weight_ih_l0, rnn.weight_hh_l0, rnn.bias_ih_l0, rnn.bias_hh_l0)

print("output shape:", tuple(out_ref.shape), "  h_n shape:", tuple(h_ref.shape))
print("outputs match:", torch.allclose(out_me, out_ref, atol=1e-6))
print("final state matches:", torch.allclose(h_me, h_ref[0], atol=1e-6))
for name, p in rnn.named_parameters():
    print(f"  {name:14s} {tuple(p.shape)}")

The number of parameters, $H d + H^2 + 2H$, does **not** depend on the sequence length:
the same cell handles sequences of length 5 or 5 000.

**Reading out.** For a many-to-one task we put a classifier on the last state
$\mathbf{h}_T$ (`output[:, -1]`); for an aligned many-to-many task we apply the
classifier to every state (`output`, shape `(B, T, H)` — `nn.Linear` acts on the
last dimension, so this needs no reshaping).

## 3. Backpropagation through time (BPTT)

Because the unrolled RNN is just a deep computation graph, gradients are computed
with ordinary backpropagation — applied to the unrolled graph this is called
**backpropagation through time**. Let the total loss be $\mathcal{L} = \sum_{t=1}^T \ell_t$,
where $\ell_t$ depends on $\mathbf{h}_t$. Writing $\mathbf{a}_t = W_{ih}\mathbf{x}_t + W_{hh}\mathbf{h}_{t-1} + \mathbf{b}$
for the pre-activation, the chain rule gives

$$\frac{\partial \mathcal{L}}{\partial W_{hh}}
= \sum_{t=1}^{T}\sum_{k=1}^{t}
\frac{\partial \ell_t}{\partial \mathbf{h}_t}\,
\underbrace{\Bigg(\prod_{j=k+1}^{t} \frac{\partial \mathbf{h}_j}{\partial \mathbf{h}_{j-1}}\Bigg)}_{\text{how step } k \text{ influences step } t}
\frac{\partial^{+} \mathbf{h}_k}{\partial W_{hh}},
\qquad
\frac{\partial \mathbf{h}_j}{\partial \mathbf{h}_{j-1}} = \operatorname{diag}\!\big(1 - \mathbf{h}_j^{2}\big)\, W_{hh},$$

where $\partial^+$ denotes the "immediate" derivative that treats $\mathbf{h}_{k-1}$ as a constant.
The contribution of an input $k$ steps in the past to the loss at time $t$ is multiplied by a
**product of $t-k$ Jacobians**. That product is the heart of the matter.

### Why gradients vanish or explode

**Linear case.** Drop the nonlinearity: $\mathbf{h}_t = W\mathbf{h}_{t-1}$. Then
$\partial \mathbf{h}_t/\partial \mathbf{h}_k = W^{t-k}$. If $W = Q\Lambda Q^{-1}$ is
diagonalisable with eigenvalues $\lambda_i$,

$$W^{n} = Q\,\Lambda^{n}\,Q^{-1}, \qquad \Lambda^n = \operatorname{diag}(\lambda_1^n,\dots,\lambda_H^n).$$

Components along eigenvectors with $|\lambda_i|<1$ shrink like $|\lambda_i|^n$ and those with
$|\lambda_i|>1$ grow like $|\lambda_i|^n$ — **exponentially in the time lag**. Only the
spectral radius $\rho(W) = \max_i|\lambda_i|$ matters in the long run: gradients
vanish if $\rho(W)<1$ and explode if $\rho(W)>1$. The knife-edge $\rho(W) = 1$ is the only
regime where information neither dies nor blows up.

**With the tanh.** Since $0 < \tanh'(a) \le 1$, the diagonal factor can only shrink things:

$$\Big\lVert \prod_{j=k+1}^{t} \operatorname{diag}(1-\mathbf{h}_j^2)\,W_{hh}\Big\rVert
\;\le\; \big(\sigma_{\max}(W_{hh})\big)^{t-k},$$

so $\sigma_{\max}(W_{hh}) < 1$ is *sufficient* for vanishing gradients, and $\sigma_{\max}(W_{hh}) > 1$ is
*necessary* for exploding ones (Pascanu, Mikolov & Bengio, 2013). Saturated units ($|h|\approx 1$)
make vanishing even worse.

Let us first look at the linear case: the norm of $W^n\mathbf{v}$ for random matrices
rescaled to a given spectral radius.

In [ ]:
Hs = 64
G = rng.normal(size=(Hs, Hs)) / np.sqrt(Hs)
G = G / np.max(np.abs(np.linalg.eigvals(G)))      # spectral radius exactly 1
v = rng.normal(size=Hs); v /= np.linalg.norm(v)

plt.figure(figsize=(6, 4))
for rho in [0.8, 0.95, 1.0, 1.05, 1.2]:
    W = rho * G
    norms, u = [], v.copy()
    for n in range(60):
        norms.append(np.linalg.norm(u))
        u = W @ u
    plt.semilogy(norms, label=f"$\\rho(W)$ = {rho}")
plt.xlabel("number of steps $n$"); plt.ylabel("$\\|W^n v\\|$")
plt.title("Linear recurrence: exponential decay or growth"); plt.legend(fontsize=8); plt.show()

Now the real thing. We build tanh RNNs whose recurrent weights are drawn from
$\mathcal{N}(0, g^2/H)$, so that the spectral radius is approximately the **gain** $g$
(circular law), feed a random sequence of length $T=100$, and backpropagate a loss
that depends only on the **last** output. The gradient with respect to the input
$\mathbf{x}_t$ tells us how strongly step $t$ can still influence the loss:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{x}_t} = W_{ih}^\top \operatorname{diag}(1-\mathbf{h}_t^2)\,\frac{\partial\mathcal{L}}{\partial\mathbf{h}_t}.$$

In [ ]:
def input_grad_norms(model, T=100, d=8, seed=0):
    """||dL/dx_t|| for every t, where L depends only on the last output of `model`."""
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(1, T, d, generator=g).requires_grad_(True)
    out, _ = model(x)                       # works for nn.RNN, nn.GRU and nn.LSTM
    out[:, -1].sum().backward()
    return x.grad[0].norm(dim=-1).numpy()

def make_rnn(gain, d=8, H=64, seed=0):
    torch.manual_seed(seed)
    m = nn.RNN(d, H, batch_first=True)
    with torch.no_grad():
        m.weight_hh_l0.normal_(0.0, gain / math.sqrt(H))
    return m

plt.figure(figsize=(7, 4))
for gain in [0.5, 0.9, 1.5, 3.0]:
    gn = input_grad_norms(make_rnn(gain))
    plt.semilogy(np.maximum(gn, 1e-30), label=f"tanh RNN, gain {gain}")
plt.xlabel("time step $t$ of the input"); plt.ylabel("$\\|\\partial \\mathcal{L} / \\partial x_t\\|$")
plt.title("Gradient reaching early inputs ($T$ = 100, loss at $t$ = 100)")
plt.legend(fontsize=8); plt.show()

Read the plot from **right to left** (from the loss backwards in time). With small gain
the gradient decays exponentially: inputs a few dozen steps back receive essentially
no learning signal, so the network **cannot learn long-range dependencies** even if
they exist in the data. With a large gain the tanh units saturate and the dynamics
become chaotic; the gradient no longer decays reliably and can instead become huge
for particular inputs and parameter values. In practice:

* **Exploding** gradients are easy to detect (the loss spikes or becomes `nan`) and are
  cured by **gradient clipping** (next section).
* **Vanishing** gradients are silent — training just fails to pick up long-range
  structure. They require an **architectural** fix: the gated cells of Section 5.

**Truncated BPTT.** For very long sequences (e.g. a whole book) we split the
sequence into chunks, carry the hidden state forward between chunks but call
`h = h.detach()` so that backpropagation stops at the chunk boundary. This bounds
memory and compute at the price of not learning dependencies longer than the chunk
(which vanishing gradients would prevent anyway).

## 4. Gradient clipping

An exploding gradient corresponds to a "cliff" in the loss landscape: a region where
the loss changes extremely steeply. A plain gradient step there catapults the
parameters far away, undoing a lot of training. **Clipping by norm** rescales the
whole gradient vector $\mathbf{g}$ (all parameters concatenated) when it is too long:

$$\mathbf{g} \leftarrow \begin{cases} \mathbf{g} & \text{if } \lVert\mathbf{g}\rVert \le c,\\[2pt]
c\,\dfrac{\mathbf{g}}{\lVert\mathbf{g}\rVert} & \text{otherwise.}\end{cases}$$

The **direction** of the step is unchanged; only its length is capped at $c$. (Clipping
each coordinate separately, "clipping by value", changes the direction and is less common.)
In PyTorch it is one line between `backward()` and `step()`:

```python
loss.backward()
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
```

`clip_grad_norm_` returns the norm **before** clipping, which is a very useful quantity
to log: occasional spikes are normal for RNNs, a steadily growing norm is a warning sign.
You will implement it yourself in the exercises. Note that clipping does nothing against
*vanishing* gradients — you cannot rescale a signal that is not there.

## 5. Gated cells: LSTM and GRU

### The LSTM

The **long short-term memory** (Hochreiter & Schmidhuber, 1997; forget gate by Gers et al., 2000)
adds a second state, the **cell state** $\mathbf{c}_t$, which is updated *additively*
and controlled by three **gates** with values in $(0,1)$:

$$
\begin{aligned}
\mathbf{i}_t &= \sigma\big(W_{ii}\mathbf{x}_t + W_{hi}\mathbf{h}_{t-1} + \mathbf{b}_i\big) && \text{input gate}\\
\mathbf{f}_t &= \sigma\big(W_{if}\mathbf{x}_t + W_{hf}\mathbf{h}_{t-1} + \mathbf{b}_f\big) && \text{forget gate}\\
\mathbf{g}_t &= \tanh\big(W_{ig}\mathbf{x}_t + W_{hg}\mathbf{h}_{t-1} + \mathbf{b}_g\big) && \text{candidate values}\\
\mathbf{o}_t &= \sigma\big(W_{io}\mathbf{x}_t + W_{ho}\mathbf{h}_{t-1} + \mathbf{b}_o\big) && \text{output gate}\\[4pt]
\mathbf{c}_t &= \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \mathbf{g}_t && \text{additive cell update}\\
\mathbf{h}_t &= \mathbf{o}_t \odot \tanh(\mathbf{c}_t).
\end{aligned}
$$

PyTorch stacks the four weight blocks in the order **(i, f, g, o)**:
`weight_ih` has shape $(4H, d)$, `weight_hh` has shape $(4H, H)$.

**Why it helps.** Along the cell-state path, ignoring the (weaker) dependence of the gates on
$\mathbf{h}_{t-1}$,

$$\frac{\partial \mathbf{c}_t}{\partial \mathbf{c}_{t-1}} \approx \operatorname{diag}(\mathbf{f}_t)
\quad\Longrightarrow\quad
\frac{\partial \mathbf{c}_t}{\partial \mathbf{c}_{k}} \approx \operatorname{diag}\Big(\prod_{j=k+1}^{t}\mathbf{f}_j\Big).$$

There is **no repeated multiplication by a weight matrix and no squashing derivative** —
only by the forget gates, which the network *learns* to keep close to 1 for units that should
remember. This is the "constant error carousel" of the original paper, and it is the same idea
as the identity shortcut in a residual network (Lecture 6): an additive path lets gradients
flow unchanged across many steps / layers.

A practical consequence: if the forget gate starts at $\sigma(0) = 0.5$, the cell path *also*
decays like $0.5^n$ at initialisation. A common trick (Gers et al., 2000; Jozefowicz et al., 2015)
is to initialise the forget-gate bias to a positive value (1–3) so that the network starts out remembering.

### The GRU

The **gated recurrent unit** (Cho et al., 2014) merges cell and hidden state and uses two gates:

$$
\begin{aligned}
\mathbf{r}_t &= \sigma\big(W_{ir}\mathbf{x}_t + W_{hr}\mathbf{h}_{t-1} + \mathbf{b}_r\big) && \text{reset gate}\\
\mathbf{z}_t &= \sigma\big(W_{iz}\mathbf{x}_t + W_{hz}\mathbf{h}_{t-1} + \mathbf{b}_z\big) && \text{update gate}\\
\mathbf{n}_t &= \tanh\big(W_{in}\mathbf{x}_t + \mathbf{b}_{in} + \mathbf{r}_t \odot (W_{hn}\mathbf{h}_{t-1} + \mathbf{b}_{hn})\big) && \text{candidate}\\
\mathbf{h}_t &= (1-\mathbf{z}_t)\odot \mathbf{n}_t + \mathbf{z}_t \odot \mathbf{h}_{t-1}.
\end{aligned}
$$

The last line is a learned **convex combination** of old state and candidate: when
$\mathbf{z}_t \approx 1$ the state is copied forward unchanged, again giving an (almost)
identity Jacobian. GRUs have three weight blocks instead of four and often perform
on par with LSTMs; neither dominates the other across tasks.

Let us compare the parameter counts and then repeat the gradient-flow experiment.

In [ ]:
for cls in [nn.RNN, nn.GRU, nn.LSTM]:
    m = cls(8, 64, batch_first=True)
    print(f"{cls.__name__:5s}: {sum(p.numel() for p in m.parameters()):6d} parameters")

def set_forget_bias(lstm, value):
    """Set the forget-gate bias of a single-layer nn.LSTM (gate order i, f, g, o)."""
    Hh = lstm.hidden_size
    with torch.no_grad():
        lstm.bias_ih_l0[Hh:2 * Hh].fill_(value)
        lstm.bias_hh_l0[Hh:2 * Hh].fill_(0.0)

torch.manual_seed(0)
lstm_default = nn.LSTM(8, 64, batch_first=True)
torch.manual_seed(0)
lstm_open = nn.LSTM(8, 64, batch_first=True)
set_forget_bias(lstm_open, 3.0)

plt.figure(figsize=(7, 4))
plt.semilogy(np.maximum(input_grad_norms(make_rnn(0.9)), 1e-30), label="tanh RNN, gain 0.9")
plt.semilogy(np.maximum(input_grad_norms(lstm_default), 1e-30), label="LSTM, default init")
plt.semilogy(np.maximum(input_grad_norms(lstm_open), 1e-30), label="LSTM, forget bias = 3")
plt.xlabel("time step $t$ of the input"); plt.ylabel("$\\|\\partial \\mathcal{L} / \\partial x_t\\|$")
plt.title("Gradient flow at initialisation"); plt.legend(fontsize=8); plt.show()

With the forget gate initialised open ($\sigma(3)\approx 0.95$), the gradient reaching early
inputs decays roughly like $0.95^{n}$ instead of something like $0.5^n$ — orders of
magnitude more signal 100 steps back. And unlike the RNN, the LSTM can *learn* to push
the forget gate even closer to 1 for units that need to remember.

## 6. Demo: a long-range memory task

**Task.** Each input is a sequence of $T$ random tokens from an alphabet of $K=4$ symbols
(one-hot encoded). The first position carries an extra "marker" bit. After reading the
whole sequence, the network must output **the first token**. All later tokens are
distractors. Chance accuracy is $1/K = 25\%$.

This is a many-to-one problem whose only relevant information lies $T$ steps in the past
— exactly where vanishing gradients bite. We train a tanh RNN and an LSTM (forget bias 3)
of the same hidden size with the same budget for increasing $T$.

In [ ]:
K = 4

def memory_batch(B, T, gen):
    tokens = torch.randint(0, K, (B, T), generator=gen)
    x = F.one_hot(tokens, K).float()                 # (B, T, K)
    marker = torch.zeros(B, T, 1)
    marker[:, 0] = 1.0
    return torch.cat([x, marker], dim=-1), tokens[:, 0]   # inputs (B, T, K+1), targets (B,)

class SeqClassifier(nn.Module):
    """Recurrent encoder + linear classifier on the last hidden state (many-to-one)."""
    def __init__(self, cell, d, H, n_classes):
        super().__init__()
        self.rnn = {"rnn": nn.RNN, "gru": nn.GRU, "lstm": nn.LSTM}[cell](d, H, batch_first=True)
        self.head = nn.Linear(H, n_classes)

    def forward(self, x):
        out, _ = self.rnn(x)            # (B, T, H)
        return self.head(out[:, -1])    # (B, n_classes)

def train_memory(cell, T, steps=500, H=32, seed=0):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    model = SeqClassifier(cell, K + 1, H, K)
    if cell == "lstm":
        set_forget_bias(model.rnn, 3.0)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for step in range(steps):
        x, y = memory_batch(64, T, gen)
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    x, y = memory_batch(2000, T, gen)
    with torch.no_grad():
        return (model(x).argmax(-1) == y).float().mean().item()

lengths = [5, 15, 30, 60]
results = {"rnn": [], "lstm": []}
t0 = time.time()
for T_ in lengths:
    for cell in results:
        results[cell].append(train_memory(cell, T_))
    print(f"T = {T_:3d}   RNN acc {results['rnn'][-1]:.3f}   LSTM acc {results['lstm'][-1]:.3f}")
print(f"({time.time() - t0:.0f} s)")

plt.figure(figsize=(6, 4))
plt.plot(lengths, results["rnn"], "o-", label="tanh RNN")
plt.plot(lengths, results["lstm"], "s-", label="LSTM")
plt.axhline(1 / K, color="gray", ls="--", label="chance")
plt.xlabel("sequence length $T$"); plt.ylabel("test accuracy"); plt.ylim(0, 1.05)
plt.title("Remember the first token"); plt.legend(); plt.show()

Both models have the *capacity* to solve the task at every length — a single hidden unit
that latches the first token would do. The difference is **trainability**: the vanilla
RNN's accuracy drops towards chance as $T$ grows, because the gradient that would teach it
to store the first token has vanished by the time it reaches step 1. The LSTM keeps
a much stronger signal on its cell-state path and stays well above chance for far longer.
With a fixed training budget even the LSTM eventually struggles as $T$ grows — gating
makes long-range credit assignment *much easier*, not free.
(Exact numbers depend on the seed and the training budget; the gap is what matters.)

## 7. Demo: a character-level language model

A **language model** assigns a probability to a sequence by the chain rule,

$$p(c_1,\dots,c_T) = \prod_{t=1}^T p(c_t \mid c_1,\dots,c_{t-1}),$$

and an RNN is a natural way to parameterise each factor: the hidden state $\mathbf{h}_{t-1}$
summarises the prefix and a softmax layer outputs the distribution of the next character.
Training is an aligned many-to-many problem: the target sequence is the input shifted by one
position, and the loss is the average cross-entropy (the negative log-likelihood per character).

We train on a short text written for this course (no downloads needed). With only
a few thousand characters the model will mostly learn spelling, common words and
punctuation habits — but that is enough to see the mechanics.

In [ ]:
text = """Neural networks have a long and uneven history. In the nineteen forties, McCulloch and Pitts described an idealised neuron that summed its inputs and fired when the sum crossed a threshold. They showed that networks of such units could compute any logical function, but they said nothing about how the weights should be found. A decade later, Rosenblatt built the perceptron, a machine that learned its weights from examples, and the newspapers promised that it would soon walk, talk and see. It did not. In nineteen sixty nine, Minsky and Papert proved that a single layer of perceptrons cannot even compute the exclusive or of two inputs, and interest in neural networks faded for more than a decade.
The revival came from an old idea used in a new way. If a network is built from smooth functions, the chain rule tells us how the loss changes when any weight changes, and the gradient of every weight can be computed in one backward sweep through the network. This procedure, now called backpropagation, was discovered several times, and it became famous in the middle of the nineteen eighties when Rumelhart, Hinton and Williams showed that it allowed hidden units to learn useful internal representations. Around the same time, recurrent networks were proposed for sequences: a network that reads one symbol at a time and keeps a hidden state that summarises everything it has seen so far.
Recurrent networks were elegant but hard to train. Hochreiter showed in his thesis that the error signal flowing backwards through time either shrinks or grows exponentially with the number of steps, so that a plain recurrent network cannot learn to connect events that are far apart. His answer, developed with Schmidhuber, was the long short-term memory, a cell with gates that decide what to write, what to keep and what to read, and with an additive memory path along which the error can travel without fading.
For many years, neural networks remained one tool among many. Support vector machines and graphical models had cleaner theory, and most practitioners preferred them. The situation changed when three things came together: large labelled datasets, graphics processors that could multiply large matrices quickly, and a collection of practical tricks such as better initialisation, rectified linear units, dropout and normalisation. In twenty twelve, a deep convolutional network won an image recognition competition by a wide margin, and within a few years deep networks had become the standard approach in vision, speech and language.
In language, recurrent networks with long short-term memory cells were the state of the art for several years. They translated sentences by reading the source into a single vector and then writing the target one word at a time. That single vector was a bottleneck, and attention was introduced to let the decoder look back at every word of the source. A few years later, researchers asked whether the recurrence was needed at all. The transformer kept the attention and removed the recurrence, so that every position could be processed in parallel. Today the largest language models are transformers trained on enormous collections of text, yet the questions that troubled the early researchers remain: how to make learning stable, how to carry information across long distances, and how to know what a network has actually learned.
"""
chars = sorted(set(text))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]
print(f"{len(text)} characters, vocabulary of {V}: {''.join(chars)!r}")

def get_batch(split, B, L, gen):
    src = train_data if split == "train" else val_data
    ix = torch.randint(0, len(src) - L - 1, (B,), generator=gen).tolist()
    x = torch.stack([src[i:i + L] for i in ix])
    y = torch.stack([src[i + 1:i + L + 1] for i in ix])     # targets: shifted by one
    return x, y

class CharLSTM(nn.Module):
    def __init__(self, V, E=32, H=128):
        super().__init__()
        self.emb = nn.Embedding(V, E)
        self.lstm = nn.LSTM(E, H, batch_first=True)
        self.head = nn.Linear(H, V)

    def forward(self, idx, state=None):
        out, state = self.lstm(self.emb(idx), state)     # (B, L, H)
        return self.head(out), state                     # logits (B, L, V)

torch.manual_seed(0)
gen = torch.Generator().manual_seed(0)
lm = CharLSTM(V)
set_forget_bias(lm.lstm, 1.0)
opt = torch.optim.Adam(lm.parameters(), lr=3e-3)
L_ctx, steps = 64, 800
train_losses, val_losses, grad_norms = [], [], []
t0 = time.time()
for step in range(steps):
    x, y = get_batch("train", 32, L_ctx, gen)
    logits, _ = lm(x)
    loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
    opt.zero_grad()
    loss.backward()
    grad_norms.append(nn.utils.clip_grad_norm_(lm.parameters(), 1.0).item())
    opt.step()
    train_losses.append(loss.item())
    if step % 100 == 0 or step == steps - 1:
        with torch.no_grad():
            xv, yv = get_batch("val", 32, L_ctx, gen)
            val_losses.append((step, F.cross_entropy(lm(xv)[0].reshape(-1, V), yv.reshape(-1)).item()))
        print(f"step {step:4d}  train loss {loss.item():.3f}  val loss {val_losses[-1][1]:.3f}")
print(f"({time.time() - t0:.0f} s)  uniform guessing would give a loss of ln V = {math.log(V):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(train_losses, alpha=0.6, label="train (per batch)")
axes[0].plot(*zip(*val_losses), "o-", label="validation")
axes[0].set_xlabel("step"); axes[0].set_ylabel("cross-entropy (nats / char)"); axes[0].legend()
axes[0].set_title("Character-level LSTM")
axes[1].plot(grad_norms); axes[1].axhline(1.0, color="k", ls="--", label="clipping threshold")
axes[1].set_xlabel("step"); axes[1].set_ylabel("gradient norm before clipping"); axes[1].legend()
axes[1].set_title("Gradient norms")
plt.tight_layout(); plt.show()

The validation text is unseen; with so little data the model overfits quickly (the
training loss keeps falling while the validation loss flattens out). The right-hand plot
shows how often clipping was active: whenever the curve is above the dashed line, the
step was shortened.

### Sampling with a temperature

To generate text we feed a prompt, then repeatedly **sample** the next character from the
predicted distribution and feed it back in (carrying the hidden state, so each step costs $O(1)$).
A **temperature** $\tau > 0$ rescales the logits before the softmax:

$$p_\tau(c) = \frac{\exp(z_c/\tau)}{\sum_{c'}\exp(z_{c'}/\tau)} .$$

$\tau = 1$ is the model's own distribution; $\tau \to 0$ approaches greedy (argmax) decoding,
which is safe but repetitive; $\tau > 1$ flattens the distribution and produces more
surprising — and more garbled — text.

In [ ]:
@torch.no_grad()
def sample(model, prompt, n_chars, temperature=1.0, seed=0):
    gen = torch.Generator().manual_seed(seed)
    idx = torch.tensor([[stoi[c] for c in prompt]])         # (1, len(prompt))
    logits, state = model(idx)
    out = list(prompt)
    for _ in range(n_chars):
        probs = F.softmax(logits[0, -1] / temperature, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1, generator=gen)   # shape (1,)
        out.append(itos[nxt.item()])
        logits, state = model(nxt.view(1, 1), state)
    return "".join(out)

lm.eval()
for tau in [0.3, 0.7, 1.0, 1.5]:
    print(f"--- temperature {tau} ---")
    print(sample(lm, "The ", 250, temperature=tau))
    print()

The effect of the temperature on a single next-character distribution:

In [ ]:
with torch.no_grad():
    ctx = torch.tensor([[stoi[c] for c in "neural netw"]])
    z = lm(ctx)[0][0, -1]
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
top = torch.argsort(z, descending=True)[:12]
for ax, tau in zip(axes, [0.5, 1.0, 2.0]):
    p = F.softmax(z / tau, dim=-1)
    ent = -(p * torch.log(p + 1e-12)).sum().item()
    ax.bar([{" ": "␣", "\n": "\\n"}.get(itos[i], itos[i]) for i in top.tolist()], p[top].numpy())
    ax.set_title(f"$\\tau$ = {tau}  (entropy {ent:.2f} nats)")
axes[0].set_ylabel("probability")
plt.suptitle("Next character after 'neural netw'", y=1.03); plt.show()

## 8. Practicalities: padding, packing and bidirectional RNNs

### Batches of sequences with different lengths

Real sequences have different lengths. To put them in one tensor we **pad** them to the
longest length. But an RNN that reads padding tokens keeps updating its state, so the
"last" state of a short sequence is polluted by the padding. Two remedies:

1. Run on the padded batch and **gather** the output at each sequence's true last position
   (and mask padded positions out of the loss for many-to-many tasks).
2. **Pack** the batch with `nn.utils.rnn.pack_padded_sequence`. A packed sequence tells the
   RNN the true lengths, so it skips the padding entirely; `h_n` is then the state after
   each sequence's real last element. With `enforce_sorted=False` PyTorch sorts and unsorts
   the batch for you.

In [ ]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

torch.manual_seed(0)
rnn = nn.RNN(3, 4, batch_first=True)
seqs = [torch.randn(5, 3), torch.randn(2, 3), torch.randn(4, 3)]
lengths = torch.tensor([len(s) for s in seqs])
padded = pad_sequence(seqs, batch_first=True)            # (3, 5, 3), zero-padded
print("padded batch shape:", tuple(padded.shape), " lengths:", lengths.tolist())

with torch.no_grad():
    # Reference: run each sequence on its own.
    h_individual = torch.stack([rnn(s.unsqueeze(0))[1][0, 0] for s in seqs])
    # Naive: run on the padded batch and take h_n.
    _, h_naive = rnn(padded)
    # Packed.
    packed = pack_padded_sequence(padded, lengths, batch_first=True, enforce_sorted=False)
    out_packed, h_packed = rnn(packed)
    out_unpacked, out_lengths = pad_packed_sequence(out_packed, batch_first=True)

print("naive h_n correct per sequence: ",
      [torch.allclose(h_naive[0, i], h_individual[i], atol=1e-6) for i in range(3)])
print("packed h_n correct per sequence:",
      [torch.allclose(h_packed[0, i], h_individual[i], atol=1e-6) for i in range(3)])
print("unpacked output shape:", tuple(out_unpacked.shape), " (padded positions are zeros)")

Only the longest sequence gets the right final state in the naive version; packing fixes
all of them.

### Bidirectional RNNs

When the whole sequence is available at once (tagging, classification — but **not**
language modelling or real-time processing), we can run one RNN left-to-right and another
right-to-left and concatenate their states,
$\mathbf{h}_t = [\overrightarrow{\mathbf{h}}_t ; \overleftarrow{\mathbf{h}}_t]$, so that the
representation of position $t$ depends on both its past and its future.

In [ ]:
birnn = nn.GRU(3, 4, batch_first=True, bidirectional=True)
out, h_n = birnn(torch.randn(2, 6, 3))
print("output:", tuple(out.shape), "  -> (B, T, 2H): forward and backward states concatenated")
print("h_n:   ", tuple(h_n.shape), "  -> (num_directions, B, H)")
# The forward direction's last state is at the END of the output, the backward one's at the START:
print(torch.allclose(out[:, -1, :4], h_n[0]), torch.allclose(out[:, 0, 4:], h_n[1]))

Stacking layers (`num_layers=2`) — feeding the state sequence of one RNN as the input
sequence of the next — is also common; dropout between layers (`dropout=...`) is the
usual regulariser.

### Sequence-to-sequence and its bottleneck

For translation the input and output lengths differ. The classic **encoder–decoder**
architecture (Sutskever et al., 2014; Cho et al., 2014) uses one RNN to read the source
into its final state $\mathbf{h}_T$ and a second RNN, initialised with that state, to
generate the target one token at a time (feeding back its own predictions; during training,
the ground-truth previous token is fed instead — **teacher forcing**).

Everything the decoder knows about the source must squeeze through the fixed-size vector
$\mathbf{h}_T$. For long sentences this **bottleneck** hurts badly. The fix — letting the
decoder look back at *all* encoder states and choose what is relevant at each step — is
called **attention**, and it is where Lecture 8 begins.

## Summary

| Concept | Key equation / fact |
|---|---|
| Vanilla RNN | $\mathbf{h}_t = \tanh(W_{ih}\mathbf{x}_t + W_{hh}\mathbf{h}_{t-1} + \mathbf{b})$, weights shared over time |
| BPTT | gradient from step $t$ to step $k$ multiplied by $\prod_{j=k+1}^t \operatorname{diag}(1-\mathbf{h}_j^2)W_{hh}$ |
| Vanishing / exploding | product behaves like $\rho(W_{hh})^{t-k}$: exponential in the time lag |
| Gradient clipping | $\mathbf{g} \leftarrow \mathbf{g}\cdot\min(1, c/\lVert\mathbf{g}\rVert)$ — fixes exploding, not vanishing |
| LSTM | additive cell $\mathbf{c}_t = \mathbf{f}_t\odot\mathbf{c}_{t-1} + \mathbf{i}_t\odot\mathbf{g}_t$, $\partial\mathbf{c}_t/\partial\mathbf{c}_{t-1}\approx\operatorname{diag}(\mathbf{f}_t)$ |
| GRU | $\mathbf{h}_t = (1-\mathbf{z}_t)\odot\mathbf{n}_t + \mathbf{z}_t\odot\mathbf{h}_{t-1}$ |
| Language model | predict $c_{t+1}$ from $c_{\le t}$; sample with temperature $\tau$ |
| Variable lengths | pad + pack (`pack_padded_sequence`) or mask |

**Further reading.**
Goodfellow, Bengio & Courville, *Deep Learning*, ch. 10;
Hochreiter & Schmidhuber (1997), *Long short-term memory*;
Bengio, Simard & Frasconi (1994), *Learning long-term dependencies with gradient descent is difficult*;
Pascanu, Mikolov & Bengio (2013), *On the difficulty of training recurrent neural networks*;
Cho et al. (2014), *Learning phrase representations using RNN encoder–decoder for statistical machine translation*;
Jozefowicz, Zaremba & Sutskever (2015), *An empirical exploration of recurrent network architectures*;
Karpathy (2015), *The unreasonable effectiveness of recurrent neural networks* (blog post).